# Transformer Foundations, Part 2: Attention, Position, and RoPE

> **Guiding question:** How can one token retrieve useful information from another, and how does attention learn whether two tokens are one step or ten steps apart?

## 0. The Challenge: Identity Is Not Context

Part 1 produced one embedding row at each position:

```text
the   cat   sat   on   the   mat
```

Each row says what token appeared. No row yet knows what the other positions contain. Attention creates that communication path.

The first attempt will deliberately omit position. It will expose the next failure: if the input rows are shuffled, plain self-attention shuffles its outputs in exactly the same way. Nothing inside the operation knows which position was first or last.

```mermaid
flowchart LR
    A["Token embeddings"] --> B["Q/K/V projections"]
    B --> C["Compare Q with K"]
    C --> D["Softmax routing weights"]
    D --> E["Blend V"]
    E --> F["Contextual token vectors"]
    P["Position"] --> C
```

| Step | Failure first | Minimal fix |
|---|---|---|
| Raw attention | tokens cannot exchange information | weighted retrieval |
| No position | a shuffle only shuffles the outputs | attach position |
| Position before projection | relative gaps need not stay explicit | rotate Q and K with RoPE |
| Raw dot products | wide vectors make softmax saturate | divide scores by square root of head width |

## Fresh-Kernel Recap from Part 1

This notebook rebuilds a tiny vocabulary and a frozen four-dimensional teaching map so every attention score can be inspected. The map is deliberately hand-authored and is not evidence about a trained model's semantic geometry.

Production models use trainable embeddings learned with the rest of the network. The information path is the same; only the width and learned coordinates change.

In [ ]:
# Imports and deterministic teaching state
import math
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

SEED = 11
np.random.seed(SEED)
torch.manual_seed(SEED)
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white"})

In [ ]:
# One familiar sentence and an inspectable content map
TOKENS = ["the", "cat", "sat", "on", "the", "mat"]
VOCAB = {"the": 0, "cat": 1, "sat": 2, "on": 3, "mat": 4}
CONTENT = {
    "the": [0.05, 0.05, 0.05, 0.05],
    "cat": [0.95, 0.75, 0.15, 0.10],
    "sat": [0.20, 0.20, 0.95, 0.70],
    "on":  [0.10, 0.10, 0.45, 0.35],
    "mat": [0.90, 0.25, 0.10, 0.10],
}
embedding_matrix = torch.tensor([CONTENT[token] for token in VOCAB], dtype=torch.float32)
TOKEN_IDS = torch.tensor([VOCAB[token] for token in TOKENS], dtype=torch.long)
token_vectors = embedding_matrix[TOKEN_IDS]
SEQ_LEN, D_INPUT = token_vectors.shape

print(f"tokens={TOKENS}")
print(f"IDs shape={tuple(TOKEN_IDS.shape)}; embeddings shape={tuple(token_vectors.shape)}")
print("Teaching axes are inspectable scaffolding; a trained model learns its own coordinates.")

---

## 1. Minimal Attention: Score, Normalize, Retrieve

Use `cat` as the query. In the crude first version, every embedding serves directly as its query, key, and value.

```text
score:      how well does cat match each key?
softmax:    turn scores into positive weights that sum to one
retrieve:   blend the value vectors using those weights
```

This is a soft lookup: `cat` can retrieve a little information from several positions instead of selecting exactly one.

**Predict:** Which object-like value will receive more weight: `mat` or `sat`?

![Conceptual attention flow: cat forms a query, compares with every key, and retrieves a weighted mixture of values](images/02-attention-soft-retrieval.jpg)

*Conceptual map only: route thickness illustrates unequal compatibility; the executable cell below computes the actual scores and weights.*

In [ ]:
# Minimal attention, revealed in the same order the mechanism is discovered
query_index = TOKENS.index("cat")
query = token_vectors[query_index]
scores = token_vectors @ query / math.sqrt(D_INPUT)
weights = torch.softmax(scores, dim=0)
weighted_values = weights[:, None] * token_vectors
context_vector = weighted_values.sum(dim=0)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.7))
axes[0].scatter(range(SEQ_LEN), np.zeros(SEQ_LEN), s=420, color="#d1d5db", edgecolor="#6b7280")
for index, token in enumerate(TOKENS):
    axes[0].text(index, 0, token, ha="center", va="center")
axes[0].scatter(query_index, 0, s=560, color="#fbbf24", edgecolor="#92400e")
axes[0].set_title("1. Pick the query: cat"); axes[0].axis("off")
axes[1].bar(TOKENS, scores.numpy(), color="#2a6f97")
axes[1].set_title("2. Compare with every key"); axes[1].set_ylabel("scaled score")
axes[2].bar(TOKENS, weights.numpy(), color="#d97706")
axes[2].set_title("3. Softmax: weights sum to one"); axes[2].set_ylabel("routing weight")
axes[3].bar([f"d{i}" for i in range(D_INPUT)], context_vector.numpy(), color="#15803d")
axes[3].set_title("4. Blend the value vectors"); axes[3].set_ylabel("context feature")
for axis in axes[1:3]:
    axis.tick_params(axis="x", rotation=25)
plt.suptitle("Minimal attention: ask, compare, normalize, retrieve")
plt.tight_layout(); plt.show()

# Live reveal: the polished storyboard above remains visible in static exports.
fig, axis = plt.subplots(figsize=(8, 3.8))


def reveal_attention(stage: int):
    axis.clear()
    if stage == 0:
        axis.scatter(range(SEQ_LEN), np.zeros(SEQ_LEN), s=420, color="#d1d5db", edgecolor="#6b7280")
        for index, token in enumerate(TOKENS):
            axis.text(index, 0, token, ha="center", va="center")
        axis.scatter(query_index, 0, s=560, color="#fbbf24", edgecolor="#92400e")
        axis.set_ylim(-0.5, 0.5); axis.axis("off"); axis.set_title("1. cat asks a question")
    elif stage == 1:
        axis.bar(TOKENS, scores.numpy(), color="#2a6f97"); axis.set_title("2. compare cat with every key")
    elif stage == 2:
        axis.bar(TOKENS, weights.numpy(), color="#d97706"); axis.set_ylim(0, 1); axis.set_title("3. softmax creates routing weights")
    else:
        axis.bar([f"d{i}" for i in range(D_INPUT)], context_vector.numpy(), color="#15803d"); axis.set_title("4. weighted values become context")


animation = FuncAnimation(fig, reveal_attention, frames=4, interval=1100, repeat=True)
plt.close(fig)
display(HTML(animation.to_jshtml(default_mode="loop")))

print("weights sum to", round(weights.sum().item(), 6))
print("highest non-self weight:", TOKENS[torch.topk(weights, 2).indices[1].item()])
print("context vector:", np.round(context_vector.numpy(), 3).tolist())
print("Complaint: attention communicated, but every step ignored which row came first.")

#### Attention communicated, but it never saw position

The query retrieved a weighted mixture of values. No recurrence or fixed-size sentence summary was required.

But the calculation used only token vectors. If the rows are permuted, the same comparisons occur in a different row order. This property is called **permutation equivariance**: shuffle the inputs, and the outputs shuffle to match.

In [ ]:
# Prove that content-only self-attention is permutation equivariant
def content_only_attention(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    score_matrix = x @ x.T / math.sqrt(x.shape[-1])
    attention_weights = torch.softmax(score_matrix, dim=-1)
    return attention_weights @ x, attention_weights


original_output, original_weights = content_only_attention(token_vectors)
permutation = torch.tensor([5, 4, 3, 2, 1, 0])
shuffled_vectors = token_vectors[permutation]
shuffled_output, shuffled_weights = content_only_attention(shuffled_vectors)
equivariance_error = (shuffled_output - original_output[permutation]).abs().max().item()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(original_weights.numpy(), ax=axes[0], cmap="Blues", vmin=0, vmax=1,
        xticklabels=TOKENS, yticklabels=TOKENS)
sns.heatmap(shuffled_weights.numpy(), ax=axes[1], cmap="Blues", vmin=0, vmax=1,
        xticklabels=[TOKENS[i] for i in permutation], yticklabels=[TOKENS[i] for i in permutation])
axes[0].set_title("Original row order"); axes[1].set_title("Shuffled row order")
for axis in axes:
    axis.set_xlabel("Key"); axis.set_ylabel("Query")
plt.tight_layout(); plt.show()

assert equivariance_error < 1e-6
print(f"PASS: shuffle-equivariance error={equivariance_error:.8f}")
print("Attention exchanged content, but content alone supplied no notion of first, next, or last.")

---

## 2. Add Position Before Attention

The original Transformer added a fixed position pattern to each token vector. Think of several clock hands moving at different speeds: their combined angles give every position a distinct signature.

```text
token vector at position 0 + position pattern 0
token vector at position 1 + position pattern 1
...
```

The repeated `the` tokens now enter attention differently because they occupy different rows.

**Predict:** Will the two occurrences of `the` remain identical after the position pattern is added?

In [ ]:
# Additive sinusoidal position, shown as a pattern rather than a derivation
def sinusoidal_position(sequence_length: int, width: int) -> torch.Tensor:
    positions = torch.arange(sequence_length, dtype=torch.float32).unsqueeze(1)
    pair_indices = torch.arange(0, width, 2, dtype=torch.float32)
    speeds = torch.exp(-math.log(10000.0) * pair_indices / width)
    table = torch.zeros(sequence_length, width)
    table[:, 0::2] = torch.sin(positions * speeds)
    table[:, 1::2] = torch.cos(positions * speeds)
    return table


position_table = sinusoidal_position(SEQ_LEN, D_INPUT)
positioned_vectors = token_vectors + position_table
repeated_the_difference = (positioned_vectors[0] - positioned_vectors[4]).norm().item()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
sns.heatmap(position_table.numpy(), ax=axes[0], cmap="RdBu_r", center=0,
        xticklabels=[f"d{i}" for i in range(D_INPUT)], yticklabels=range(SEQ_LEN))
sns.heatmap(positioned_vectors.numpy(), ax=axes[1], cmap="RdBu_r", center=0,
        xticklabels=[f"d{i}" for i in range(D_INPUT)], yticklabels=TOKENS)
axes[0].set_title("Position pattern"); axes[1].set_title("Token + position")
axes[0].set_ylabel("Position"); axes[1].set_ylabel("Token position")
plt.tight_layout(); plt.show()

assert repeated_the_difference > 0
print(f"Repeated 'the' distance after adding position={repeated_the_difference:.4f}")
print("PASS: token identity stayed present, and position made the occurrences distinct.")

#### A nagging question before we move on

Additive position fixes the immediate order failure. The two occurrences of `the` are finally different.

That heatmap looks convincing. It also hides a problem: position is mixed with content **before** learned query and key projections. The model can learn to preserve relative structure, but nothing guarantees that a clean distance pattern survives the mixing and projection.

**Predict:** After random content is added and one query projection is applied, will nearby positions remain as easy to recognize as they were in the pure position table?

In [ ]:
# One changed path: pure position versus content + position through a query projection
torch.manual_seed(SEED)
D_POSITION_DEMO, POSITION_STEPS = 16, 12
pure_position = sinusoidal_position(POSITION_STEPS, D_POSITION_DEMO)
random_content = torch.randn(POSITION_STEPS, D_POSITION_DEMO)
query_projection = torch.randn(D_POSITION_DEMO, D_POSITION_DEMO) / math.sqrt(D_POSITION_DEMO)
projected_mixture = (random_content + pure_position) @ query_projection.T


def cosine_similarity_matrix(matrix: torch.Tensor) -> torch.Tensor:
    normalized = matrix / (matrix.norm(dim=-1, keepdim=True) + 1e-9)
    return normalized @ normalized.T


def distance_readability(similarity: torch.Tensor) -> float:
    size = similarity.shape[0]
    closeness = torch.tensor([[-abs(i - j) for j in range(size)] for i in range(size)], dtype=torch.float32)
    return float(np.corrcoef(closeness.flatten().numpy(), similarity.flatten().numpy())[0, 1])


pure_similarity = cosine_similarity_matrix(pure_position)
projected_similarity = cosine_similarity_matrix(projected_mixture)
pure_readability = distance_readability(pure_similarity)
projected_readability = distance_readability(projected_similarity)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for axis, matrix, title in [
    (axes[0], pure_similarity, "Pure position pattern\nclean distance bands"),
    (axes[1], projected_similarity, "After content + random Q projection\nrelative pattern is harder to read"),
]:
    sns.heatmap(matrix.numpy(), ax=axis, cmap="RdBu_r", vmin=-1, vmax=1, square=True)
    axis.set_xlabel("Position"); axis.set_ylabel("Position"); axis.set_title(title)
plt.tight_layout(); plt.show()

assert pure_readability > projected_readability
print(f"distance readability: pure={pure_readability:.3f}; projected mixture={projected_readability:.3f}")
print("PASS: additive position remains valid, but its relative pattern is not guaranteed to stay explicit after projection.")
print("Complaint: put position where attention actually compares Q with K.")

#### The complaint points to RoPE

The experiment did **not** prove that sinusoidal position fails. It proved a narrower point: adding position at the input does not guarantee that relative distance remains explicit in the score geometry.

RoPE answers that complaint by applying position after Q and K are projected, immediately before their dot product.

![Additive encoding mixes position into token states before Q, K, and V projection, while RoPE rotates projected Q and K immediately before comparison; in both designs, projected V bypasses scoring and joins the attention weights only at the weighted sum](images/02-additive-position-vs-rope.jpg)

*Conceptual wiring map, not measured output. In both paths, Q and K choose the route while projected V carries the retrieved content. The experiment above measures why moving position into the score path can keep relative geometry explicit.*

---

## 3. RoPE: Turn Query and Key Pairs Like Dials

After Q and K are projected and split into heads, RoPE treats each adjacent feature pair as a tiny 2D dial. Position decides how far each dial turns; different pairs turn at different speeds.

```text
position 0: no turn
position 1: one step at this pair's speed
position 5: five steps at this pair's speed
```

Rotation changes direction, not vector length. The useful trick appears when a query at one position meets a key at another: shift both together and their absolute turns change, but their relative gap does not.

**Important boundary:** RoPE makes relative gaps distinguishable. It does not guarantee that farther tokens always receive lower attention; token content and learned projections still control the score.

In [ ]:
# Attempt 1: the crudest RoPE picture - one dial, one pair, one position
D_ROPE = 6
ROPE_SPEEDS = np.array([1.0, 0.35, 0.08], dtype=np.float32)
attempt_position = 5
attempt_angle = attempt_position * ROPE_SPEEDS[0]
fig, axis = plt.subplots(figsize=(3.3, 3.3), subplot_kw={"aspect": "equal"})
axis.add_patch(plt.Circle((0, 0), 1, fill=False, color="#9ca3af"))
axis.arrow(0, 0, math.cos(attempt_angle), math.sin(attempt_angle), width=0.035,
           head_width=0.16, length_includes_head=True, color="#d97706")
axis.axhline(0, color="#d1d5db", lw=0.7); axis.axvline(0, color="#d1d5db", lw=0.7)
axis.set(xlim=(-1.2, 1.2), ylim=(-1.2, 1.2), title="Attempt 1: pair 0 at position 5")
axis.set_xticks([]); axis.set_yticks([])
plt.tight_layout(); plt.show()
print("Complaint 1: one query/key vector contains several pairs, not one dial.")

# Attempt 2: stack every pair and compare several positions in one static storyboard.
STORYBOARD_POSITIONS = [0, 1, 5, 12]
fig, axes = plt.subplots(3, len(STORYBOARD_POSITIONS), figsize=(11, 7), subplot_kw={"aspect": "equal"})
for pair_index, row_axes in enumerate(axes):
    for column, position in enumerate(STORYBOARD_POSITIONS):
        axis = row_axes[column]
        angle = position * ROPE_SPEEDS[pair_index]
        axis.add_patch(plt.Circle((0, 0), 1, fill=False, color="#9ca3af"))
        axis.arrow(0, 0, math.cos(angle), math.sin(angle), width=0.035,
                   head_width=0.16, length_includes_head=True, color="#d97706")
        axis.set(xlim=(-1.2, 1.2), ylim=(-1.2, 1.2))
        axis.axhline(0, color="#d1d5db", lw=0.7); axis.axvline(0, color="#d1d5db", lw=0.7)
        axis.set_xticks([]); axis.set_yticks([])
        if pair_index == 0:
            axis.set_title(f"position {position}")
        if column == 0:
            axis.set_ylabel(f"pair {pair_index}\nspeed {ROPE_SPEEDS[pair_index]:.2f}")
fig.suptitle("Attempt 2: every pair, across several token positions")
plt.tight_layout(); plt.show()
print("Complaint 2: snapshots show the pattern; motion should make the different speeds unmistakable.")

# Attempt 3: animate the same measured rotations. The storyboard remains for static exports.
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), subplot_kw={"aspect": "equal"})


def draw_rope_frame(position: int):
    for pair_index, axis in enumerate(axes):
        axis.clear()
        angle = position * ROPE_SPEEDS[pair_index]
        axis.add_patch(plt.Circle((0, 0), 1, fill=False, color="#9ca3af"))
        axis.arrow(0, 0, math.cos(angle), math.sin(angle), width=0.035,
                   head_width=0.16, length_includes_head=True, color="#d97706")
        axis.set(xlim=(-1.2, 1.2), ylim=(-1.2, 1.2))
        axis.axhline(0, color="#d1d5db", lw=0.7); axis.axvline(0, color="#d1d5db", lw=0.7)
        axis.set_title(f"pair {pair_index}: speed={ROPE_SPEEDS[pair_index]:.2f}")
        axis.set_xticks([]); axis.set_yticks([])
    fig.suptitle(f"Attempt 3: RoPE dials at token position {position}")


animation = FuncAnimation(fig, draw_rope_frame, frames=24, interval=140, repeat=True)
plt.close(fig)
display(HTML(animation.to_jshtml(default_mode="loop")))
print("Payoff: one position turns every pair, and each pair carries position at its own scale.")

In [ ]:
# Measure the two RoPE invariants
def rope_rotate(vector: np.ndarray, position: int, speeds: np.ndarray) -> np.ndarray:
    rotated = np.array(vector, dtype=np.float32).copy()
    for pair_index, speed in enumerate(speeds):
        angle = position * speed
        cosine, sine = math.cos(angle), math.sin(angle)
        even, odd = rotated[2 * pair_index], rotated[2 * pair_index + 1]
        rotated[2 * pair_index] = even * cosine - odd * sine
        rotated[2 * pair_index + 1] = even * sine + odd * cosine
    return rotated


rng = np.random.default_rng(SEED)
query_base, key_base = rng.normal(size=D_ROPE), rng.normal(size=D_ROPE)
norm_errors = [abs(np.linalg.norm(rope_rotate(query_base, p, ROPE_SPEEDS)) - np.linalg.norm(query_base))
           for p in [0, 1, 5, 17]]
dot_a = rope_rotate(query_base, 2, ROPE_SPEEDS) @ rope_rotate(key_base, 7, ROPE_SPEEDS)
dot_b = rope_rotate(query_base, 5, ROPE_SPEEDS) @ rope_rotate(key_base, 10, ROPE_SPEEDS)
shift_error = abs(dot_a - dot_b)

assert max(norm_errors) < 1e-5 and shift_error < 1e-5
print(f"PASS 1 - maximum vector-length change={max(norm_errors):.8f}")
print(f"PASS 2 - equal-gap dot-product change={shift_error:.8f}")
print("The pairs (2, 7) and (5, 10) share gap 5 even though their absolute positions differ.")

---

## 4. Q, K, and V: Ask, Advertise, Deliver

Raw embeddings should not have to use one geometry for every job. Learned projections give each token three roles:

| Projection | Question it answers |
|---|---|
| Query | What information am I looking for? |
| Key | What kind of information do I advertise? |
| Value | What information do I deliver if selected? |

RoPE rotates Q and K because they decide the routing score. It does not rotate V because V carries the retrieved content.

In [ ]:
# Learned Q/K/V projections: first see the geometry, then inspect every feature
D_MODEL = 8
torch.manual_seed(SEED)
input_projection = nn.Linear(D_INPUT, D_MODEL, bias=False)
W_Q = nn.Linear(D_MODEL, D_MODEL, bias=False)
W_K = nn.Linear(D_MODEL, D_MODEL, bias=False)
W_V = nn.Linear(D_MODEL, D_MODEL, bias=False)

x = input_projection(token_vectors) + sinusoidal_position(SEQ_LEN, D_MODEL)
Q, K, V = W_Q(x), W_K(x), W_V(x)
score_matrix = Q @ K.T / math.sqrt(D_MODEL)
attention_weights = torch.softmax(score_matrix, dim=-1)
context = attention_weights @ V

fig, axes = plt.subplots(1, 4, figsize=(15, 3.7))
for axis, matrix, title, color in [
    (axes[0], x.detach(), "Input geometry", "#6b7280"),
    (axes[1], Q.detach(), "Q: what each token seeks", "#2563eb"),
    (axes[2], K.detach(), "K: what each token advertises", "#d97706"),
    (axes[3], V.detach(), "V: what each token sends", "#15803d"),
]:
    points = matrix[:, :2].numpy()
    axis.scatter(points[:, 0], points[:, 1], s=70, color=color)
    for token_index, token in enumerate(TOKENS):
        axis.text(points[token_index, 0], points[token_index, 1], f" {token}", fontsize=8)
    axis.axhline(0, color="#d1d5db", lw=0.7); axis.axvline(0, color="#d1d5db", lw=0.7)
    axis.set_title(title); axis.set_xlabel("feature 0"); axis.set_ylabel("feature 1")
plt.suptitle("The same token cloud is stretched and rotated into three job-specific spaces")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for axis, matrix, title in zip(axes, [Q.detach(), K.detach(), V.detach()], ["Queries", "Keys", "Values"]):
    sns.heatmap(matrix.numpy(), ax=axis, cmap="RdBu_r", center=0, cbar=False,
                xticklabels=[f"d{i}" for i in range(D_MODEL)], yticklabels=TOKENS)
    axis.set_title(title)
plt.tight_layout(); plt.show()

print(f"x {tuple(x.shape)} -> Q/K/V {tuple(Q.shape)}")
print(f"Q @ K.T -> scores {tuple(score_matrix.shape)} -> context {tuple(context.shape)}")
print("Payoff: similarity and delivered content no longer have to share one geometry.")

## 5. Put RoPE Inside the Real Score Path

The dials now move from the mental model into attention:

```text
project Q and K -> pair their features -> rotate by position
-> compare rotated Q with rotated K -> softmax -> retrieve unrotated V
```

The next comparison holds the projections fixed and changes only whether Q and K receive RoPE.

In [ ]:
# Same Q/K/V, one variable changed: RoPE on or off
def build_rope_cache(sequence_length: int, width: int, base: float = 10000.0):
    pair_indices = torch.arange(0, width, 2, dtype=torch.float32)
    frequencies = 1.0 / (base ** (pair_indices / width))
    angles = torch.outer(torch.arange(sequence_length, dtype=torch.float32), frequencies)
    return angles.cos(), angles.sin()


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    even, odd = x[..., 0::2], x[..., 1::2]
    return torch.stack((even * cos - odd * sin, even * sin + odd * cos), dim=-1).flatten(-2)


cos, sin = build_rope_cache(SEQ_LEN, D_MODEL)
Q_rot, K_rot = apply_rope(Q, cos, sin), apply_rope(K, cos, sin)
raw_weights = torch.softmax(Q @ K.T / math.sqrt(D_MODEL), dim=-1)
rope_weights = torch.softmax(Q_rot @ K_rot.T / math.sqrt(D_MODEL), dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for axis, matrix, title in zip(axes, [raw_weights.detach(), rope_weights.detach()], ["Same projections, no RoPE", "Same projections, RoPE on Q and K"]):
    sns.heatmap(matrix.numpy(), ax=axis, cmap="mako", vmin=0, vmax=max(raw_weights.max(), rope_weights.max()).item(),
        xticklabels=TOKENS, yticklabels=TOKENS)
    axis.set_title(title); axis.set_xlabel("Key"); axis.set_ylabel("Query")
plt.tight_layout(); plt.show()

norm_error = (Q.norm(dim=-1) - Q_rot.norm(dim=-1)).abs().max().item()
assert norm_error < 1e-5
print(f"Maximum Q norm change after RoPE={norm_error:.8f}")
print("PASS: RoPE changed routing geometry without changing query-vector lengths.")

---

## 6. Why Scale the Scores?

As query/key width grows, random dot products spread farther from zero. Softmax then becomes nearly one-hot before the model has learned a good reason to be confident. Near a saturated choice, useful gradients shrink.

Dividing by the square root of head width keeps the score scale comparable across widths. The expression is compact, but the measured effect matters more:

`attention = softmax(QK^T / sqrt(head_width)) V`

**Predict:** As width grows, which curve will stay stable: unscaled or scaled peak probability?

In [ ]:
# Measure both symptoms: false confidence and shrinking softmax sensitivity
torch.manual_seed(SEED)
widths = [4, 16, 64, 256]
unscaled_peaks, scaled_peaks = [], []
unscaled_sensitivity, scaled_sensitivity = [], []
for width in widths:
    queries = torch.randn(1500, 1, width)
    keys = torch.randn(1500, 12, width)
    raw_scores = (queries * keys).sum(dim=-1)
    unscaled_probabilities = torch.softmax(raw_scores, dim=-1)
    scaled_probabilities = torch.softmax(raw_scores / math.sqrt(width), dim=-1)
    unscaled_peaks.append(unscaled_probabilities.max(dim=-1).values.mean().item())
    scaled_peaks.append(scaled_probabilities.max(dim=-1).values.mean().item())
    unscaled_sensitivity.append((unscaled_probabilities * (1 - unscaled_probabilities)).mean().item())
    scaled_sensitivity.append((scaled_probabilities * (1 - scaled_probabilities)).mean().item())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(widths, unscaled_peaks, "o-", label="unscaled", color="#b91c1c")
axes[0].plot(widths, scaled_peaks, "o-", label="scaled", color="#15803d")
axes[0].set_xscale("log", base=2); axes[0].set_ylim(0, 1)
axes[0].set_xlabel("Query/key width"); axes[0].set_ylabel("Mean maximum probability")
axes[0].set_title("Symptom 1: width creates false confidence"); axes[0].legend()
axes[1].plot(widths, unscaled_sensitivity, "o-", label="unscaled", color="#b91c1c")
axes[1].plot(widths, scaled_sensitivity, "o-", label="scaled", color="#15803d")
axes[1].set_xscale("log", base=2)
axes[1].set_xlabel("Query/key width"); axes[1].set_ylabel("Mean softmax sensitivity p(1-p)")
axes[1].set_title("Symptom 2: saturation weakens the learning signal"); axes[1].legend()
plt.tight_layout(); plt.show()

assert unscaled_peaks[-1] > scaled_peaks[-1] + 0.25
assert scaled_sensitivity[-1] > unscaled_sensitivity[-1] * 2
print("unscaled peaks:", [round(value, 3) for value in unscaled_peaks])
print("scaled peaks:  ", [round(value, 3) for value in scaled_peaks])
print("PASS: scaling controls confidence and preserves a more useful softmax slope at large width.")

---

## Chapter 2 Checkpoint

```text
embeddings
-> Q/K/V projections
-> RoPE rotates Q and K by position
-> scaled QK comparison
-> softmax routing weights
-> weighted V retrieval
-> contextual token vectors
```

| Question | Measured answer |
|---|---|
| Can tokens retrieve one another? | minimal attention produced normalized routing weights and a blended value |
| Does plain attention know order? | no; shuffled inputs produced identically shuffled outputs |
| Can additive position distinguish repeated tokens? | yes; the two `the` rows diverged |
| What does RoPE preserve? | vector length and selected same-gap dot products |
| Why rotate Q and K? | they determine routing; V carries retrieved content |
| Why scale scores? | scaled peak confidence stayed stable as width grew |

**Your turn:** Change the minimal-attention query from `cat` to `mat`. Predict the strongest non-self route before rerunning the score and weight plots.

**Next:** [Part 3 — The Complete Transformer Block](03-transformer-block.ipynb) adds multiple attention heads, the per-token FFN, residual paths, normalization, vocabulary logits, loss, and one complete backward pass.